# Phase 1: Data Generation using OpenRouter API (Serverless)
Notebook này sử dụng **OpenRouter API** để sinh dữ liệu.
Mã nguồn sử dụng cấu trúc HTTP Request thô (Raw HTTP Request) không thông qua thư viện trung gian. Để đảm bảo tốc độ chạy Đa Luồng (Song song), chúng ta dùng `aiohttp` thay vì `requests` (vì `requests` sẽ khóa luồng và làm máy chạy rất chậm).

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q datasets aiohttp jsonlines tqdm nest_asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply() # Cho phép chạy asyncio trong Jupyter Notebook

import os
import json
import asyncio
import aiohttp # Thư viện gọi HTTP Request bất đồng bộ (như thư viện requests nhưng xịn hơn)
import jsonlines
from datasets import load_dataset
from tqdm.asyncio import tqdm

## 2. Cấu hình API và Tham số

In [ ]:
OPENROUTER_API_KEY = "sk-or-v1-YOUR_TOKEN_HERE" # THAY TOKEN OPENROUTER CỦA BẠN VÀO ĐÂY

# Model Llama 3.3 70B miễn phí trên OpenRouter
MODEL_ID = "meta-llama/llama-3.3-70b-instruct:free" 

NUM_ROLLOUTS = 4
MAX_NEW_TOKENS = 1024

# Giới hạn số request chạy ĐỒNG THỜI
CONCURRENT_REQUESTS = 3

## 3. Tải và Lọc Dataset

In [ ]:
print("Loading dataset...")
try:
    ds = load_dataset("open-thoughts/OpenThoughts-114k", "metadata", split="train")
except Exception as e:
    print(f"Lỗi cấu hình 'metadata': {e}")
    ds = load_dataset("open-thoughts/OpenThoughts-114k", "default", split="train")

print("Filtering for math questions...")
def is_math(example):
    return str(example.get('domain', '')).lower().strip() == 'math'

ds_math = ds.filter(is_math)

# Lấy ngẫu nhiên 100 câu
SAMPLE_SIZE = 100
ds_sample = ds_math.shuffle(seed=42).select(range(min(SAMPLE_SIZE, len(ds_math))))
print(f"Loaded {len(ds_sample)} math samples.")

## 4. Khởi tạo Logic Bất Đồng Bộ bằng HTTP Request

In [ ]:
SYSTEM_PROMPT = (
    "Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process "
    "before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle "
    "of analysis, summarizing, exploration, reassessment, reflection, backtracing, and iteration to develop well-reasoned answers."
)

# Semaphore dùng để khóa luồng
sem = asyncio.Semaphore(CONCURRENT_REQUESTS)

async def generate_single_rollout(question, rollout_idx):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        "max_tokens": MAX_NEW_TOKENS,
        "temperature": 0.7,
        "top_p": 0.95
    }
    
    max_retries = 8 
    for attempt in range(max_retries):
        try:
            async with sem:
                async with aiohttp.ClientSession() as session:
                    async with session.post(
                        "https://openrouter.ai/api/v1/chat/completions",
                        headers=headers,
                        data=json.dumps(payload)
                    ) as response:
                        if response.status == 200:
                            data = await response.json()
                            return data["choices"][0]["message"]["content"]
                        elif response.status == 429:
                            raise Exception("Rate Limit 429")
                        else:
                            error_text = await response.text()
                            raise Exception(f"Lỗi HTTP {response.status}: {error_text}")
                            
        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "rate limit" in error_msg:
                wait_time = (attempt + 1) * 8  
                print(f"[Rate Limit] Chờ {wait_time}s... (Thử lại lần {attempt+1})")
                await asyncio.sleep(wait_time)
            else:
                print(f"[Lỗi Khác] {e}. Chờ 3s... (Thử lại lần {attempt+1})")
                await asyncio.sleep(3)
                
    return "[LỖI: Không thể lấy được phản hồi từ OpenRouter sau nhiều lần thử]"

async def process_question(item):
    question = item.get('problem', '')
    if not question:
        return None
        
    # Tạo 4 task gọi API cùng lúc cho 1 câu hỏi
    tasks = [generate_single_rollout(question, i) for i in range(NUM_ROLLOUTS)]
    
    # Chờ cả 4 task hoàn thành
    rollouts = await asyncio.gather(*tasks)
    
    return {
        "question": question,
        "ground_truth": item.get('ground_truth_solution', ''),
        "deepseek_reference_reasoning": item.get('deepseek_reasoning', ''),
        "deepseek_reference_solution": item.get('deepseek_solution', ''),
        "generated_rollouts": rollouts
    }

## 5. Chạy Tiến Trình Ghi Dữ Liệu

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    output_file = "/content/drive/MyDrive/Data_Phase1/phase1_openrouter_generated.jsonl"
else:
    output_file = "phase1_openrouter_generated.jsonl"

async def main():
    print(f"Bắt đầu sinh dữ liệu qua Raw HTTP OpenRouter API. Giới hạn: {CONCURRENT_REQUESTS} requests cùng lúc...")
    
    with jsonlines.open(output_file, mode='a') as writer:
        tasks = [process_question(item) for item in ds_sample]
        
        for coro in tqdm.as_completed(tasks, total=len(tasks), desc="Processing via OpenRouter"):
            result = await coro
            if result:
                writer.write(result)
                
    print(f"\nHoàn thành! Đã lưu an toàn vào {output_file}")

await main()